## What is Tool:
外部插件，能夠讓 claude 調用，擁有以下優點:
- **Real-time Information** : Access current data that wasn't available during Claude's training
- **External System Integration** : Connect Claude to databases, APIs, and other services
- **Dynamic Responses** : Provide answers based on the latest available information
- **Structured Interaction** : Claude knows exactly what information it needs and how to ask for it

In [24]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [25]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [26]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

## 1. Tool Function
撰寫 tool function 供 model 呼叫，需要遵守以下 guidelines:
1. **Use descriptive names** : Both your function name and parameter names should clearly indicate their purpose
2. **Validate inputs** : Check that required parameters aren't empty or invalid, and raise errors when they are
3. **Provide meaningful error messages** : Claude can see error messages and might retry the function call with corrected parameters

![](./figure/03/Tool_Function_1.png)

In [27]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

## 2. Tool Schema
將寫好的 tool function 轉換成一種 *類似 JSON 格式* 的 schema，讓 LLM 更好理解 tool 的內容
必備 3 個要素 :
1. **name** - A clear, descriptive name for your tool (like "get_weather")
2. **description** - What the tool does, when to use it, and what it returns
3. **input_schema** - The actual JSON schema describing the function's arguments

其中 Description 對 LLM 的理解至關重要，以下是要遵守的撰寫原則:
- Aim for 3-4 sentences explaining what the tool does
- Describe when Claude should use it
- Explain what kind of data it returns
- Provide detailed descriptions for each argument

![](./figure/03/Tool_Schema_1.png)
![](./figure/03/Tool_Schema_2.png)

### 如何寫一個好的 schema? step 如下:
1. Copy your tool function code
2. Go to Claude and ask it to write a JSON schema for tool calling 
    - 範例: ```"Write a valid JSON schema spec for the purposes of tool calling for this function. Follow the best practices listed in the attached documentation."```
3. Include the [Anthropic documentation on tool use](https://platform.claude.com/docs/en/agents-and-tools/tool-use/overview) as context (複製整夜內容當作 attachment 給 claude)
4. Let Claude generate a properly formatted schema following best practices
5. Use the pattern of function_name followed by function_name_schema to keep your schemas organized and easy to match with their corresponding functions.

![](./figure/03/Tool_Schema_3.png)

In [28]:
def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

# Use the pattern of function_name followed by function_name_schema to keep your schemas organized and easy to match with their corresponding functions.

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
}

### 為後續做準備，進行型別轉換
import and use the ```ToolParam``` type from the *Anthropic library*

In [29]:
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "Returns the current date and time formatted according to the specified format",
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": "A string specifying the format of the returned datetime. Uses Python's strftime format codes.",
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []
    }
})

## 3. How to call Claude with schema

### 3-1. Handling message blocks
如何將 tool 進要傳遞給 model 的資訊中?
在 ``` client.messages.create() ``` 中將剛剛撰寫好的 *schema* 放入在 ``` tools ``` 這個 **list** argument 中即可

(這邊故意不使用先前撰寫好的 helper function，單獨處理比較清楚)

In [30]:
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],
)

print(response)

Message(id='msg_01345mkNezPuBCgmZV6voanj', container=None, content=[ToolUseBlock(id='toolu_01S2wWASTdSrmu5MffN56uCK', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=619, output_tokens=63, server_tool_use=None, service_tier='standard'))


在上一格的輸出內容中可以看到模型回傳的資料，content 為主要內容，其中包含了兩種 Block 
1. **"Text" Block** : model 的回覆內容，project 先前的對話回傳 content 裡都只有這種 Block
2. **"Tooluse" Block** :  當 model 要使用 tool 時，會將需求放置此處，並傳遞給 server (在此處就是你手上這台電腦)，裡面包含以下內容:
    - An ID for tracking the tool call
    - The name of the function to call (like "get_current_datetime")
    - Input parameters formatted as a dictionary
    - The type designation "tool_use"

![](./figure/03/Message_Block_1.png)

Claude 並不會儲存對話記憶，因此需要將 model 先前回傳的 content 更新至 conversation history 中

後續在使用時 model 才能夠理解先前的流程，不會亂做

In [31]:
messages.append({
    "role": "assistant",
    "content": response.content
})

print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01S2wWASTdSrmu5MffN56uCK', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


### 使用 tool 流程暫時總結:
1. Send user message with tool schema to Claude
2. Receive assistant message with text block and tool use block
3. Extract tool information and execute the actual function
4. Send tool result back to Claude along with complete conversation history
5. Receive final response from Claude

## 4. 根據 Claude 回傳 content 使用 Tool 
當 server 收到 model 使用 tool 的請求時，會解析 ```ToolUseBlock```，確認模型要如何使用 tool

In [32]:
print(response)

Message(id='msg_01345mkNezPuBCgmZV6voanj', container=None, content=[ToolUseBlock(id='toolu_01S2wWASTdSrmu5MffN56uCK', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=619, output_tokens=63, server_tool_use=None, service_tier='standard'))


In [33]:
response.content[0].input

{'date_format': '%H:%M:%S'}

從上述內容可以獲得 model 要傳入 tool 的 argument，再透過 python 的定位 syntax 讓他自動匹配位置 

In [34]:
get_current_datetime(**response.content[0].input)

'16:49:51'

## 5. 將 Tool result 傳回給 model
將 result 回傳給 model 時需要將資訊放置在 **User message** 中的 **``` Tool Result Block ```** 進行傳輸，告訴 model 執行 tool 發生了什麼
其中會包含以下重要內容:
- **tool_use_id** : Must match the id of the ToolUse block that this ToolResult corresponds to
- **content** : Output from running your tool, serialized as a string
- **is_error** : True if an error occurred

![](./figure/03/Tool_Result_1.png)

以下 code 將 tool result 包裝進 user message 回傳給 claude

包裝後的 ```messages``` 會包含:
- Original user message
- Assistant message with tool use block
- User message with tool result block

In [ ]:
# 包裝前
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01S2wWASTdSrmu5MffN56uCK', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}]


In [ ]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[0].id,
        "content": "15:04:22",
        "is_error": False
    }]
})

# 包裝後
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_01S2wWASTdSrmu5MffN56uCK', caller=DirectCaller(type='direct'), input={'date_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use')]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_01S2wWASTdSrmu5MffN56uCK', 'content': '15:04:22', 'is_error': False}]}]


接著將 messages 回傳給 claude，其即可依據這些內容給予新的 response

* Note : 要特別說明，即使後續不使用 tool，仍然將 tool schema 附上較好，因為在先前的對話有使用過 tool，還是得讓現在的 model 知道有相對應的知識，不然他沒有記憶，可能會出錯 

![](./figure/03/Tool_Result_2.png)

In [37]:
after_tool_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]
)

In [38]:
print(after_tool_response)

Message(id='msg_01EgZv8hKjSCqn1PjSc1gtTj', container=None, content=[TextBlock(citations=None, text='The exact time is **15:04:22** (3:04:22 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=699, output_tokens=23, server_tool_use=None, service_tier='standard'))


### 單一 response 中使用多次 tool 該如何回傳? 

claude 可能會在一次 response 中要求使用多次 tool。在每次呼叫 tool 時 claude 都會給予一個 **```id```**，當要將結果回覆給 claude 時務必確保包含 **```id```**，這樣即使在 order 不對的情況下 claude 仍然能夠給予使用者相對應的正確回復 

![](./figure/03/Tool_multiple_Result_1.png)